# Member 4 — T2f (FLAIR) Pipeline | Kaggle
**Run cells top to bottom. Every cell prints its status.**

| Cell | What it does | Expected output |
|------|-------------|----------------|
| 1 | Setup | 4 green ✓ path checks, GPU name, patch size |
| 2 | Sanity check | 4 MRI images with coloured tumour contours |
| 3 | Model smoke test | Both archs: forward ✓ features ✓ |
| 4 | Training | Loss + Dice printed every 5 epochs |
| 5 | Training curves | Two-panel loss + Dice plot |
| 6 | Evaluation | 9 metrics table (WT/TC/ET × Dice/IoU/HD95) |
| 7 | Grad-CAM | 9 PNGs per patient × 3 patients = 27 PNGs |
| 8 | SHAP | Run overnight (SKIP_SHAP=True by default) |
| 9 | Export for M5 | Files listed with sizes |

In [1]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/wesalmagdysoliman
/kaggle/input/datasets/wesalmagdysoliman/brats-subset
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02970-100
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02821-101
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02105-105
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02365-100
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02757-100
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02514-101
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-00009-101
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02871-101
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02861-100
/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset/BraTS-GLI-02206-1

In [2]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Setup
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

print('Installing packages...')
for pkg in ['mlflow', 'shap']:
    r = subprocess.run(['pip', 'install', '-q', pkg], capture_output=True)
    print(f'  {pkg}: OK')

# ── Paths ────────────────────────────────────────────────
CODE_BASE  = Path('/kaggle/input/datasets/wesalmagdysoliman/brats-code/kaggle_upload')
SHARED_DIR = CODE_BASE / 'shared'
M4_DIR     = CODE_BASE / 'member4_T2f'
SPLITS_DIR = CODE_BASE / 'data' / 'splits'

DATA_ROOT  = Path('/kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset')
WORK       = Path('/kaggle/working')

for p in [str(CODE_BASE), str(M4_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('\nPath checks:')
all_ok = True
for label, path in [
    ('shared/',      SHARED_DIR),
    ('member4_T2f/', M4_DIR),
    ('splits/',      SPLITS_DIR),
    ('data/',        DATA_ROOT),
]:
    ok = path.exists()
    print(f'  {"✓" if ok else "✗ MISSING"} {label:20s} {path}')
    all_ok = all_ok and ok

if not all_ok:
    raise RuntimeError('Missing paths above — check dataset is attached to notebook')

# ── Seed (must be first) ─────────────────────────────────
from shared.seed import set_global_seed
set_global_seed()

# ── GPU ──────────────────────────────────────────────────
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    props    = torch.cuda.get_device_properties(0)
    VRAM     = props.total_memory / 1e9
    GPU_NAME = props.name
else:
    VRAM, GPU_NAME = 0.0, 'CPU (no GPU — training will be slow)'

# ── Hyperparameters ───────────────────────────────────────
# Defined here — we do NOT import shared.config because it
# tries to read DATA_PATH.local.txt which does not exist on Kaggle.
PATCH_SIZE   = 96 if VRAM >= 14 else 64
BATCH_SIZE   = 2
NUM_WORKERS  = 4        # Kaggle is Linux — safe to use workers
LR           = 1e-4
WEIGHT_DECAY = 1e-5
NUM_EPOCHS   = 50       # reduce to 5 for a quick smoke test
PATIENCE     = 15
VAL_EVERY    = 5
AMP_ENABLED  = True
BEST_METRIC  = 'dice_wt'
MEMBER_NAME  = 'member4_T2f'
MODALITY     = 't2f'

# Monkey-patch BEST_METRIC into shared.config so shared/trainer.py
# can import it without needing DATA_PATH.local.txt
import importlib, types
try:
    import shared.config as _cfg
    if not hasattr(_cfg, 'BEST_METRIC'):
        _cfg.BEST_METRIC = BEST_METRIC
    if not hasattr(_cfg, 'GRAD_CLIP_NORM'):
        _cfg.GRAD_CLIP_NORM = 1.0
    if not hasattr(_cfg, 'PATCH_SIZE'):
        _cfg.PATCH_SIZE = (PATCH_SIZE,) * 3
except Exception as e:
    print(f'  config patch: {e} — creating stub')
    _cfg = types.ModuleType('shared.config')
    _cfg.BEST_METRIC    = BEST_METRIC
    _cfg.GRAD_CLIP_NORM = 1.0
    _cfg.PATCH_SIZE     = (PATCH_SIZE,) * 3
    sys.modules['shared.config'] = _cfg

# ── Output dirs ───────────────────────────────────────────
CKPT_DIR   = WORK / 'checkpoints';       CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR    = WORK / 'results/T2F/figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR  = WORK / 'results/T2F/tables'; TABLE_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = WORK / 'export_for_M5';     EXPORT_DIR.mkdir(parents=True, exist_ok=True)
MLRUNS_DIR = WORK / 'mlruns';            MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
TEAM_TABLE = WORK / 'results/tables';    TEAM_TABLE.mkdir(parents=True, exist_ok=True)

print(f'\n{"═"*55}')
print(f'  GPU     : {GPU_NAME}')
print(f'  VRAM    : {VRAM:.1f} GB')
print(f'  Patch   : {PATCH_SIZE}³')
print(f'  Device  : {DEVICE}')
print(f'  Patients: {len(list(DATA_ROOT.iterdir()))} in subset')
for split in ["train","val","test"]:
    ids = [l.strip() for l in (SPLITS_DIR/f"{split}_ids.txt").read_text().splitlines() if l.strip()]
    print(f'  {split:5s}   : {len(ids)} patients')
print(f'{"═"*55}')
print('Setup OK ✓')

Installing packages...
  mlflow: OK
  shap: OK

Path checks:
  ✓ shared/              /kaggle/input/datasets/wesalmagdysoliman/brats-code/kaggle_upload/shared
  ✓ member4_T2f/         /kaggle/input/datasets/wesalmagdysoliman/brats-code/kaggle_upload/member4_T2f
  ✓ splits/              /kaggle/input/datasets/wesalmagdysoliman/brats-code/kaggle_upload/data/splits
  ✓ data/                /kaggle/input/datasets/wesalmagdysoliman/brats-subset/subset
[seed] Global seed set to 42
[config] Tesla T4 | 15.6 GB | patch=(96, 96, 96) | Kaggle=True

═══════════════════════════════════════════════════════
  GPU     : Tesla T4
  VRAM    : 15.6 GB
  Patch   : 96³
  Device  : cuda
  Patients: 351 in subset
  train   : 280 patients
  val     : 35 patients
  test    : 35 patients
═══════════════════════════════════════════════════════
Setup OK ✓


In [3]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 24.5 MB/s eta 0:00:0000:0100:01


In [4]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Sanity check
# ═══════════════════════════════════════════════════════════
import glob, nibabel as nib
from shared.preprocessing import load_and_normalise

test_ids = [l.strip() for l in (SPLITS_DIR/'test_ids.txt').read_text().splitlines() if l.strip()]
PID      = test_ids[0]
PDIR     = str(DATA_ROOT / PID)
print(f'Patient : {PID}')
print(f'Files   : {[os.path.basename(f) for f in glob.glob(PDIR+"/*.nii.gz")]}')

vols = {m: load_and_normalise(glob.glob(f'{PDIR}/*{m}*.nii*')[0])
        for m in ['t1c','t1n','t2f','t2w']}
seg  = nib.load(glob.glob(f'{PDIR}/*seg*.nii*')[0]).get_fdata()

print(f'\nVolume  : {seg.shape}')
for lbl, name in [(0,'BG'),(1,'NCR'),(2,'ED'),(3,'ET'),(4,'RC')]:
    n = (seg==lbl).sum()
    if n > 0: print(f'  label {lbl} ({name}): {n:,} ({100*n/seg.size:.3f}%)')

best_z = int((seg>0).sum(axis=(1,2)).argmax())
fig, axes = plt.subplots(1, 4, figsize=(18,5), facecolor='#111')
colours = {1:'#AA44FF', 2:'#1D9E75', 3:'#EF9F27', 4:'#DD4444'}
for ax, (mod, title) in zip(axes, [('t1c','T1c'),('t1n','T1n'),('t2f','T2f FLAIR ★'),('t2w','T2w')]):
    ax.imshow(vols[mod][best_z].T, cmap='gray', origin='lower')
    for lbl, col in colours.items():
        mask = (seg[best_z]==lbl)
        if mask.any(): ax.contour(mask.T, levels=[0.5], colors=[col], linewidths=1.5)
    ax.set_title(title, color='white'); ax.axis('off')
    if mod=='t2f':
        for s in ax.spines.values(): s.set_edgecolor('#1D9E75'); s.set_linewidth(2)

plt.suptitle(f'{PID} | axial z={best_z}', color='white', y=1.01)
plt.tight_layout()
save_p = str(FIG_DIR/'sanity_check.png')
plt.savefig(save_p, dpi=130, bbox_inches='tight', facecolor='#111')
plt.show()
print(f'Saved {save_p}')
print('Sanity check OK ✓')

2026-05-15 23:16:53.626480: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778887013.829147      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778887013.889429      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778887014.370731      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778887014.370773      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778887014.370776      57 computation_placer.cc:177] computation placer alr

Patient : BraTS-GLI-02193-104
Files   : []

Volume  : (182, 218, 182)
  label 0 (BG): 7,169,473 (99.286%)
  label 2 (ED): 36,842 (0.510%)
  label 3 (ET): 47 (0.001%)
  label 4 (RC): 14,670 (0.203%)
Saved /kaggle/working/results/T2F/figures/sanity_check.png
Sanity check OK ✓


In [5]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Model smoke test
# ═══════════════════════════════════════════════════════════
from model import build_model, BOTTLENECK_CHANNELS

P             = PATCH_SIZE
dummy         = torch.randn(2, 1, P, P, P)
expected_out  = (2, 3, P, P, P)
expected_feat = (2, BOTTLENECK_CHANNELS, P//16, P//16, P//16)

for arch in ('resunet','segresnet'):
    m      = build_model(arch, in_channels=1, out_channels=3)
    params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    out    = m(dummy)
    feat   = m.forward_features(dummy)
    assert tuple(out.shape)  == expected_out,  f'{arch} output {out.shape} ≠ {expected_out}'
    assert tuple(feat.shape) == expected_feat, f'{arch} feat   {feat.shape} ≠ {expected_feat}'
    print(f'[{m.arch_name:12s}] forward={tuple(out.shape)} ✓   features={tuple(feat.shape)} ✓   params={params:,}')

print('\nModel smoke test passed ✓')

[ResUNet3D   ] forward=(2, 3, 96, 96, 96) ✓   features=(2, 256, 6, 6, 6) ✓   params=9,924,131
[SegResNet   ] forward=(2, 3, 96, 96, 96) ✓   features=(2, 256, 6, 6, 6) ✓   params=18,796,035

Model smoke test passed ✓


In [25]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Training
# Change NUM_EPOCHS = 5 above for a quick smoke test first
# ═══════════════════════════════════════════════════════════
import mlflow, time
import torch.optim as optim
from shared.dataset import BraTSDataset, get_dataloader
from shared.trainer import (
    train_one_epoch, validate_one_epoch,
    EarlyStopper, CheckpointManager, dice_bce_loss,
)
from shared.metrics import compute_all_metrics, MetricTracker

ALL_RESULTS = {}

def run_arch(arch: str, n_epochs: int = NUM_EPOCHS) -> dict:
    ckpt_name = f'{MEMBER_NAME}_{arch}'
    model     = build_model(arch, in_channels=1, out_channels=3).to(DEVICE)
    params    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    run_name  = f'M4-T2F-{model.arch_name}-seed42'

    train_ds = BraTSDataset(
        data_root=str(DATA_ROOT), split_file=str(SPLITS_DIR/'train_ids.txt'),
        modality=MODALITY, patch_size=PATCH_SIZE,
        augment=True, patches_per_volume=4,
    )
    val_ds = BraTSDataset(
        data_root=str(DATA_ROOT), split_file=str(SPLITS_DIR/'val_ids.txt'),
        modality=MODALITY, patch_size=PATCH_SIZE,
        augment=False, full_volume=True,
    )
    train_loader = get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    val_loader   = get_dataloader(val_ds,   batch_size=1,          shuffle=False, num_workers=NUM_WORKERS)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    scaler    = torch.amp.GradScaler('cuda') if (AMP_ENABLED and DEVICE.type=='cuda') else None
    stopper   = EarlyStopper(patience=PATIENCE)
    ckpt_mgr  = CheckpointManager(str(CKPT_DIR), ckpt_name)

    train_losses, val_dices_wt, val_epochs = [], [], []
    best_dice = -1.0
    t0 = time.time()

    mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
    mlflow.set_experiment('brats-gli-2024')

    print(f'\n{"="*60}')
    print(f'  {run_name}')
    print(f'  {params:,} params | {DEVICE} | patch={PATCH_SIZE}³ | batch={BATCH_SIZE}')
    print(f'  train={len(train_ds)} patches | val={len(val_ds)} volumes')
    print(f'{"="*60}')

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({'arch':model.arch_name,'modality':MODALITY,
                           'patch':PATCH_SIZE,'lr':LR,'epochs':n_epochs,
                           'device':str(DEVICE),'vram_gb':round(VRAM,1)})

        for epoch in range(1, n_epochs+1):
            tm = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, dice_bce_loss)
            scheduler.step()
            train_losses.append(tm['loss'])
            mlflow.log_metric('train_loss', tm['loss'], step=epoch)
            print(f'Epoch {epoch:03d}/{n_epochs} | loss={tm["loss"]:.4f}', end='')

            if epoch % VAL_EVERY == 0 or epoch == n_epochs:
                vm  = validate_one_epoch(model, val_loader, DEVICE, dice_bce_loss)
                vd  = float(vm.get('dice_wt', 0.0))
                val_dices_wt.append(vd); val_epochs.append(epoch)
                for k,v in vm.items():
                    if isinstance(v,float): mlflow.log_metric(f'val_{k}', v, step=epoch)
                is_best = vd > best_dice
                if is_best: best_dice = vd
                ckpt_mgr.save(model, optimizer, epoch, vm,
                              is_best=is_best, best_metric_key=BEST_METRIC)
                print(f' | dice_wt={vd:.4f} tc={vm.get("dice_tc",0):.4f} et={vm.get("dice_et",0):.4f}'
                      + (' ← best' if is_best else ''))
                # should_stop accepts dict directly
                if stopper.should_stop(vm):
                    print(f'  Early stopping at epoch {epoch}'); break
            else:
                print()

    elapsed = time.time()-t0
    print(f'\nDone. best_dice_wt={best_dice:.4f}  ({elapsed/60:.1f} min)')
    return {'arch':model.arch_name,'best_dice_wt':best_dice,
            'train_losses':train_losses,'val_dices_wt':val_dices_wt,
            'val_epochs':val_epochs}

# for arch in ['resunet','segresnet']:
for arch in ['resunet']:
    ALL_RESULTS[arch] = run_arch(arch)

BEST_ARCH = max(ALL_RESULTS, key=lambda a: ALL_RESULTS[a]['best_dice_wt'])
print(f'\nBest arch: {BEST_ARCH} (dice_wt={ALL_RESULTS[BEST_ARCH]["best_dice_wt"]:.4f})')

2026/05/15 13:22:27 INFO mlflow.tracking.fluent: Experiment with name 'brats-gli-2024' does not exist. Creating a new experiment.



  M4-T2F-ResUNet3D-seed42
  9,924,131 params | cuda | patch=96³ | batch=2
  train=1120 patches | val=35 volumes
Epoch 001/50 | loss=1.1685
Epoch 002/50 | loss=0.9912
Epoch 003/50 | loss=0.8482
Epoch 004/50 | loss=0.7554
Epoch 005/50 | loss=0.6970[ckpt] member4_T2f_resunet_epoch005.pt saved (119.3 MB, dice=0.7993)
[ckpt] new best -> member4_T2f_resunet_best.pt
 | dice_wt=0.7993 tc=0.2827 et=0.2721 ← best
[early-stop] val Dice improved -inf -> 0.7993
Epoch 006/50 | loss=0.6649
Epoch 007/50 | loss=0.6387
Epoch 008/50 | loss=0.6220
Epoch 009/50 | loss=0.6079
Epoch 010/50 | loss=0.5998[ckpt] member4_T2f_resunet_epoch010.pt saved (119.3 MB, dice=0.8232)
[ckpt] new best -> member4_T2f_resunet_best.pt
 | dice_wt=0.8232 tc=0.2188 et=0.1697 ← best
[early-stop] val Dice improved 0.7993 -> 0.8232
Epoch 011/50 | loss=0.5885
Epoch 012/50 | loss=0.5854
Epoch 013/50 | loss=0.5789
Epoch 014/50 | loss=0.5746
Epoch 015/50 | loss=0.5701[ckpt] member4_T2f_resunet_epoch015.pt saved (119.3 MB, dice=0.8254)


KeyboardInterrupt: 

In [6]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Training
# Change NUM_EPOCHS = 5 above for a quick smoke test first
# ═══════════════════════════════════════════════════════════
import mlflow, time
import torch.optim as optim
from shared.dataset import BraTSDataset, get_dataloader
from shared.trainer import (
    train_one_epoch, validate_one_epoch,
    EarlyStopper, CheckpointManager, dice_bce_loss,
)
from shared.metrics import compute_all_metrics, MetricTracker

ALL_RESULTS = {}

def run_arch(arch: str, n_epochs: int = NUM_EPOCHS) -> dict:
    ckpt_name = f'{MEMBER_NAME}_{arch}'
    model     = build_model(arch, in_channels=1, out_channels=3).to(DEVICE)
    params    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    run_name  = f'M4-T2F-{model.arch_name}-seed42'

    train_ds = BraTSDataset(
        data_root=str(DATA_ROOT), split_file=str(SPLITS_DIR/'train_ids.txt'),
        modality=MODALITY, patch_size=PATCH_SIZE,
        augment=True, patches_per_volume=4,
    )
    val_ds = BraTSDataset(
        data_root=str(DATA_ROOT), split_file=str(SPLITS_DIR/'val_ids.txt'),
        modality=MODALITY, patch_size=PATCH_SIZE,
        augment=False, full_volume=True,
    )
    train_loader = get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    val_loader   = get_dataloader(val_ds,   batch_size=1,          shuffle=False, num_workers=NUM_WORKERS)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    scaler    = torch.amp.GradScaler('cuda') if (AMP_ENABLED and DEVICE.type=='cuda') else None
    stopper   = EarlyStopper(patience=PATIENCE)
    ckpt_mgr  = CheckpointManager(str(CKPT_DIR), ckpt_name)

    train_losses, val_dices_wt, val_epochs = [], [], []
    best_dice = -1.0
    t0 = time.time()

    mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
    mlflow.set_experiment('brats-gli-2024')

    print(f'\n{"="*60}')
    print(f'  {run_name}')
    print(f'  {params:,} params | {DEVICE} | patch={PATCH_SIZE}³ | batch={BATCH_SIZE}')
    print(f'  train={len(train_ds)} patches | val={len(val_ds)} volumes')
    print(f'{"="*60}')

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({'arch':model.arch_name,'modality':MODALITY,
                           'patch':PATCH_SIZE,'lr':LR,'epochs':n_epochs,
                           'device':str(DEVICE),'vram_gb':round(VRAM,1)})

        for epoch in range(1, n_epochs+1):
            tm = train_one_epoch(model, train_loader, optimizer, scaler, DEVICE, dice_bce_loss)
            scheduler.step()
            train_losses.append(tm['loss'])
            mlflow.log_metric('train_loss', tm['loss'], step=epoch)
            print(f'Epoch {epoch:03d}/{n_epochs} | loss={tm["loss"]:.4f}', end='')

            if epoch % VAL_EVERY == 0 or epoch == n_epochs:
                vm  = validate_one_epoch(model, val_loader, DEVICE, dice_bce_loss)
                vd  = float(vm.get('dice_wt', 0.0))
                val_dices_wt.append(vd); val_epochs.append(epoch)
                for k,v in vm.items():
                    if isinstance(v,float): mlflow.log_metric(f'val_{k}', v, step=epoch)
                is_best = vd > best_dice
                if is_best: best_dice = vd
                ckpt_mgr.save(model, optimizer, epoch, vm,
                              is_best=is_best, best_metric_key=BEST_METRIC)
                print(f' | dice_wt={vd:.4f} tc={vm.get("dice_tc",0):.4f} et={vm.get("dice_et",0):.4f}'
                      + (' ← best' if is_best else ''))
                # should_stop accepts dict directly
                if stopper.should_stop(vm):
                    print(f'  Early stopping at epoch {epoch}'); break
            else:
                print()

    elapsed = time.time()-t0
    print(f'\nDone. best_dice_wt={best_dice:.4f}  ({elapsed/60:.1f} min)')
    return {'arch':model.arch_name,'best_dice_wt':best_dice,
            'train_losses':train_losses,'val_dices_wt':val_dices_wt,
            'val_epochs':val_epochs}

# for arch in ['resunet','segresnet']:
for arch in ['segresnet']:
    ALL_RESULTS[arch] = run_arch(arch)

BEST_ARCH = max(ALL_RESULTS, key=lambda a: ALL_RESULTS[a]['best_dice_wt'])
print(f'\nBest arch: {BEST_ARCH} (dice_wt={ALL_RESULTS[BEST_ARCH]["best_dice_wt"]:.4f})')

2026/05/15 23:17:42 INFO mlflow.tracking.fluent: Experiment with name 'brats-gli-2024' does not exist. Creating a new experiment.



  M4-T2F-SegResNet-seed42
  18,796,035 params | cuda | patch=96³ | batch=2
  train=1120 patches | val=35 volumes
Epoch 001/50 | loss=1.0238
Epoch 002/50 | loss=0.9027
Epoch 003/50 | loss=0.8047
Epoch 004/50 | loss=0.7353
Epoch 005/50 | loss=0.6926[ckpt] member4_T2f_segresnet_epoch005.pt saved (225.7 MB, dice=0.8326)
[ckpt] new best -> member4_T2f_segresnet_best.pt
 | dice_wt=0.8326 tc=0.1810 et=0.1528 ← best
[early-stop] val Dice improved -inf -> 0.8326
Epoch 006/50 | loss=0.6636
Epoch 007/50 | loss=0.6422
Epoch 008/50 | loss=0.6302
Epoch 009/50 | loss=0.6158
Epoch 010/50 | loss=0.6098[ckpt] member4_T2f_segresnet_epoch010.pt saved (225.7 MB, dice=0.8174)
 | dice_wt=0.8174 tc=0.2054 et=0.1861
[early-stop] no improvement (1/15) best=0.8326 latest=0.8174
Epoch 011/50 | loss=0.6011
Epoch 012/50 | loss=0.6004
Epoch 013/50 | loss=0.5922
Epoch 014/50 | loss=0.5839
Epoch 015/50 | loss=0.5803[ckpt] member4_T2f_segresnet_epoch015.pt saved (225.7 MB, dice=0.8357)
[ckpt] new best -> member4_T2f_s

In [12]:
!zip -r /kaggle/working/all_working.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/export_for_M5/ (stored 0%)
  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/tables/ (stored 0%)
  adding: kaggle/working/results/T2F/ (stored 0%)
  adding: kaggle/working/results/T2F/tables/ (stored 0%)
  adding: kaggle/working/results/T2F/features/ (stored 0%)
  adding: kaggle/working/results/T2F/checkpoints/ (stored 0%)
  adding: kaggle/working/results/T2F/figures/ (stored 0%)
  adding: kaggle/working/results/T2F/figures/sanity_check.png (deflated 2%)
  adding: kaggle/working/results/T2F/figures/training_curves.png (deflated 14%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 73%)
  adding: kaggle/working/mlruns/ (stored 0%)
  adding: kaggle/working/mlruns/437203029534472000/ (stored 0%)
  adding: kaggle/working/mlruns/437203029534472000/2ce24efc88e0448189997fd9e745c09e/ (stored 0%)
  adding: kaggle/work

In [13]:
from IPython.display import FileLink
FileLink(r'all_working.zip')

/kaggle/working/all_working.zip

In [9]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Training curves
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14,5), facecolor='#111')
cols = {'resunet':'#1D9E75','segresnet':'#EF9F27'}

for arch, r in ALL_RESULTS.items():
    c = cols[arch]
    axes[0].plot(r['train_losses'], color=c, lw=2, label=r['arch'])
    axes[1].plot(r['val_epochs'], r['val_dices_wt'], color=c, lw=2,
                 marker='o', ms=5, label=f"{r['arch']} best={r['best_dice_wt']:.4f}")

for ax, title, yl in [(axes[0],'Training loss','Loss'),(axes[1],'Validation Dice_WT','Dice')]:
    ax.set_facecolor('#111'); ax.set_title(title,color='white')
    ax.set_xlabel('Epoch',color='white'); ax.set_ylabel(yl,color='white')
    ax.tick_params(colors='white'); ax.legend(facecolor='#1a1a1a',labelcolor='white')
    ax.axhline(0.60,color='#888',lw=1,linestyle='--',alpha=0.5,label='target 0.60')
    for s in ax.spines.values(): s.set_edgecolor('#333')

plt.suptitle('M4 Training curves', color='white', fontsize=13)
plt.tight_layout()
save_p = str(FIG_DIR/'training_curves.png')
plt.savefig(save_p, dpi=130, bbox_inches='tight', facecolor='#111')
plt.show()
print(f'Saved {save_p}')

Saved /kaggle/working/results/T2F/figures/training_curves.png


In [10]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Evaluation on test set (9 metrics)
# ═══════════════════════════════════════════════════════════
import pandas as pd
from monai.inferers import sliding_window_inference

test_ds     = BraTSDataset(
    data_root=str(DATA_ROOT), split_file=str(SPLITS_DIR/'test_ids.txt'),
    modality=MODALITY, patch_size=PATCH_SIZE, augment=False, full_volume=True,
)
test_loader = get_dataloader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)
print(f'Test volumes: {len(test_ds)}')

rows = []
for arch in ['resunet','segresnet']:
    ckpt_name = f'{MEMBER_NAME}_{arch}'
    model_e   = build_model(arch, in_channels=1, out_channels=3)
    try:
        mgr = CheckpointManager(str(CKPT_DIR), ckpt_name)
        model_e, _, ep, vd = mgr.load_best(model_e)
        model_e = model_e.to(DEVICE).eval()
        print(f'[{arch}] checkpoint: epoch={ep} val_dice_wt={vd:.4f}')
    except FileNotFoundError:
        print(f'[{arch}] no checkpoint — skipping'); continue

    tracker = MetricTracker()
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            logits = sliding_window_inference(
                images, (PATCH_SIZE,)*3, 4, model_e, overlap=0.5)
            tracker.update(compute_all_metrics(logits, labels))

    m = tracker.compute()
    print(f'[{arch}] Dice  WT={m["dice_wt"]:.4f}  TC={m["dice_tc"]:.4f}  ET={m["dice_et"]:.4f}')
    print(f'         IoU   WT={m["iou_wt"]:.4f}  TC={m["iou_tc"]:.4f}  ET={m["iou_et"]:.4f}')
    print(f'         HD95  WT={m["hd95_wt"]:.1f}   TC={m["hd95_tc"]:.1f}   ET={m["hd95_et"]:.1f} mm')
    rows.append({'member':'M4','modality':'T2f (FLAIR)',
                 'architecture':model_e.arch_name,
                 **{k:round(float(v),4) for k,v in m.items()}})

df_eval = pd.DataFrame(rows)
df_eval.to_csv(str(TABLE_DIR/'M4_test_full.csv'), index=False)

# Append to team-wide test_metrics.csv
team_csv = TEAM_TABLE / 'test_metrics.csv'
if team_csv.exists():
    existing = pd.read_csv(team_csv)
    existing = existing[existing['member'] != 'M4']
    pd.concat([existing, df_eval], ignore_index=True).to_csv(str(team_csv), index=False)
else:
    df_eval.to_csv(str(team_csv), index=False)

print(f'\nSaved M4_test_full.csv ✓')
print(f'Appended to test_metrics.csv ✓  (unlocks xai_analysis.py guard)')
print(df_eval[['architecture','dice_wt','dice_tc','dice_et','hd95_wt']].to_string(index=False))

Test volumes: 35
[resunet] no checkpoint — skipping
[segresnet] checkpoint: epoch=50 val_dice_wt=0.8489


KeyboardInterrupt: 

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Grad-CAM (per sub-region, per view, 3 patients)
# Produces 27 PNGs: gradcam_M4_{pid}_{sr}_{view}.png
# ═══════════════════════════════════════════════════════════
from shared.grad_cam_3d import GradCAM3D
from shared.preprocessing import preprocess_patient
from shared.visualization import plot_gradcam_overlay

# Load best arch for XAI
model_xai = build_model(BEST_ARCH, in_channels=1, out_channels=3)
mgr_xai   = CheckpointManager(str(CKPT_DIR), f'{MEMBER_NAME}_{BEST_ARCH}')
model_xai, _, _, _ = mgr_xai.load_best(model_xai)
model_xai.cpu().eval()
enc_layer = model_xai.get_encoder_layers()[-1]   # deepest encoder

# 3 deterministic patients: first, middle, last
n_t  = len(test_ids)
pids = [test_ids[0], test_ids[n_t//2], test_ids[-1]]
SUBREGIONS = ('wt','tc','et')
VIEWS      = ('axial','coronal','sagittal')
n_saved    = 0

for pid in pids:
    pdir       = str(DATA_ROOT / pid)
    vol_np, seg_np = preprocess_patient(pdir, MODALITY)
    vol_t      = torch.tensor(vol_np[np.newaxis]).float()   # (1,1,D,H,W)
    vol_sq     = vol_np.squeeze()    # (D,H,W)
    seg_sq     = seg_np.squeeze()    # (D,H,W)  raw integer labels

    # Build binary mask per sub-region for slice selection
    sr_masks = {
        'wt': (seg_sq==1)|(seg_sq==2)|(seg_sq==3),
        'tc': (seg_sq==1)|(seg_sq==3),
        'et':  seg_sq==3,
    }

    with GradCAM3D(model_xai, enc_layer) as cam:
        for sr in SUBREGIONS:
            heatmap = cam.generate(vol_t, target_channel=sr)   # (D,H,W) float32
            np.save(str(FIG_DIR/f'gradcam_M4_{pid}_{sr}.npy'), heatmap)

            mask = sr_masks[sr]
            bz   = int(mask.sum(axis=(1,2)).argmax()) if mask.any() else vol_sq.shape[0]//2
            by_  = int(mask.sum(axis=(0,2)).argmax()) if mask.any() else vol_sq.shape[1]//2
            bx   = int(mask.sum(axis=(0,1)).argmax()) if mask.any() else vol_sq.shape[2]//2

            view_slices = [
                ('axial',    vol_sq[bz],       heatmap[bz],       mask[bz].astype(float)),
                ('coronal',  vol_sq[:,by_,:],  heatmap[:,by_,:],  mask[:,by_,:].astype(float)),
                ('sagittal', vol_sq[:,:,bx],   heatmap[:,:,bx],   mask[:,:,bx].astype(float)),
            ]
            for view_name, v_sl, c_sl, s_sl in view_slices:
                fname = str(FIG_DIR/f'gradcam_M4_{pid}_{sr}_{view_name}.png')
                plot_gradcam_overlay(
                    vol_slice=v_sl, cam_slice=c_sl, seg_slice=s_sl,
                    title=f'M4 FLAIR | {pid[:20]} | {sr.upper()} | {view_name}',
                    save_path=fname)
                n_saved += 1

    print(f'{pid}: 9 Grad-CAM PNGs saved ✓')

print(f'\nTotal saved: {n_saved} PNGs (expected 27)')
print(f'Location: {FIG_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — SHAP (slow — ~30 min per patient)
# Set SKIP_SHAP = False and rerun when you have time
# ═══════════════════════════════════════════════════════════
SKIP_SHAP = True
SHAP_BG   = 20

if not SKIP_SHAP:
    import shap
    from scipy.stats import pearsonr

    model_xai.cpu().eval()
    for pid in pids:
        pdir     = str(DATA_ROOT / pid)
        vol_np, _ = preprocess_patient(pdir, MODALITY)
        vol_t    = torch.tensor(vol_np[np.newaxis]).float()

        # Background: spatially-shifted versions of the same volume
        bg_list = []
        for _ in range(SHAP_BG):
            shift = tuple(np.random.randint(-10,10,size=3))
            bg    = np.roll(vol_np[np.newaxis], shift, axis=(1,2,3)).astype(np.float32)
            bg_list.append(bg)
        background = torch.tensor(np.stack(bg_list)).float()

        for sr_idx, sr in enumerate(SUBREGIONS):
            class _Wrap(torch.nn.Module):
                def __init__(self, m, ch): super().__init__(); self.m=m; self.ch=ch
                def forward(self, x): return self.m(x)[:, self.ch:self.ch+1]

            explainer = shap.GradientExplainer(_Wrap(model_xai, sr_idx), background)
            sv        = explainer.shap_values(vol_t)
            if isinstance(sv, list): sv = sv[0]
            shap_map  = np.abs(sv).squeeze().astype(np.float32)
            if shap_map.max() > 0: shap_map /= shap_map.max()
            np.save(str(FIG_DIR/f'shap_M4_{pid}_{sr}.npy'), shap_map)

            cam_npy = np.load(str(FIG_DIR/f'gradcam_M4_{pid}_{sr}.npy'))
            r, _    = pearsonr(cam_npy.flatten(), shap_map.flatten())
            interp  = 'HIGH ✓' if r>0.7 else ('MODERATE' if r>0.4 else 'LOW — investigate')
            print(f'  {pid[:15]} {sr.upper()}: Pearson r={r:.3f}  [{interp}]')

        print(f'{pid}: SHAP done ✓')
else:
    print('SHAP skipped (SKIP_SHAP=True).')
    print('Set SKIP_SHAP=False and rerun this cell when you have time.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Export for M5
# Downloads from: Kaggle → Output tab → /kaggle/working/export_for_M5/
# ═══════════════════════════════════════════════════════════
import shutil, nibabel as nib

pred_dir = EXPORT_DIR/'predictions';    pred_dir.mkdir(exist_ok=True)
gcam_dir = EXPORT_DIR/'gradcam_arrays'; gcam_dir.mkdir(exist_ok=True)

# 1. Copy best checkpoint
best_ckpt = CKPT_DIR/f'{MEMBER_NAME}_{BEST_ARCH}_best.pt'
if best_ckpt.exists():
    shutil.copy2(best_ckpt, EXPORT_DIR/'member4_T2f_best.pt')
    print(f'Checkpoint: {best_ckpt.stat().st_size/1e6:.1f} MB copied ✓')

# 2. Predictions + Grad-CAM for all test patients
model_exp = build_model(BEST_ARCH, in_channels=1, out_channels=3)
mgr_exp   = CheckpointManager(str(CKPT_DIR), f'{MEMBER_NAME}_{BEST_ARCH}')
model_exp, _, ep_exp, vd_exp = mgr_exp.load_best(model_exp)
model_exp.cpu().eval()
print(f'Loaded: epoch={ep_exp} val_dice_wt={vd_exp:.4f}')

n_preds = 0
for pid in test_ids:
    pdir = str(DATA_ROOT / pid)
    try:
        vol_np, seg_np = preprocess_patient(pdir, MODALITY)
    except Exception as e:
        print(f'  {pid}: skip — {e}'); continue

    vol_t = torch.tensor(vol_np[np.newaxis]).float()
    with torch.no_grad():
        logits = sliding_window_inference(vol_t, (PATCH_SIZE,)*3, 2, model_exp, overlap=0.25)
    pred_3ch = (torch.sigmoid(logits) > 0.5).squeeze().numpy().astype(np.int16)  # (3,D,H,W)

    # Save WT prediction as NIfTI (M5 uses this for fusion)
    nib.save(nib.Nifti1Image(pred_3ch[0], np.eye(4)), str(pred_dir/f'{pid}_pred_wt.nii.gz'))
    n_preds += 1

    # Copy Grad-CAM arrays
    for sr in SUBREGIONS:
        src = FIG_DIR/f'gradcam_M4_{pid}_{sr}.npy'
        if src.exists(): shutil.copy2(src, gcam_dir/f'{pid}_{sr}_gradcam.npy')

print(f'Predictions saved: {n_preds}/{len(test_ids)} ✓')

# 3. Metrics CSV
shutil.copy2(str(TABLE_DIR/'M4_test_full.csv'), str(EXPORT_DIR/'M4_row.csv'))

# 4. Summary
print(f'\n{"="*55}')
print('  EXPORT COMPLETE — send export_for_M5/ to M5')
print(f'{"="*55}')
total_mb = 0
for f in sorted(EXPORT_DIR.rglob('*')):
    if f.is_file():
        mb = f.stat().st_size/1e6
        total_mb += mb
        print(f'  {str(f.relative_to(EXPORT_DIR)):50s} {mb:6.1f} MB')
print(f'  {"Total":50s} {total_mb:6.1f} MB')
print(f'\nDownload: Kaggle notebook → Output tab → /kaggle/working/export_for_M5/')